# Step 03: Self-attention

Tests for `03-attention/attention.py`. Write code in the `.py` file, only import and test here.

The bigram sees one token back. Attention lets every token look at all earlier tokens.

Run the cells top to bottom (Run All).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import torch
import matplotlib.pyplot as plt

sys.path.append("03-attention")
from attention import (
    VOCAB_SIZE, get_device, load_data, split_data, decode, train,
    causal_mean_loop, causal_mean_matmul, causal_mean_softmax,
    Head, MultiHeadAttention, AttentionLanguageModel,
)

device = get_device()
print("device:", device)

## 1. Averaging the past, three ways

Goal: each token gets the mean of itself and all earlier tokens, and never sees later ones. Same answer three times, ending in the softmax-mask form that attention uses.

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 4, 3)           # (B=2, T=4, C=3)

a = causal_mean_loop(x)
b = causal_mean_matmul(x)
c = causal_mean_softmax(x)
print(a.shape)                     # expect (2, 4, 3)
print(torch.allclose(a, b), torch.allclose(a, c))   # expect True True

# the weights behind it: row t says how much token t takes from each earlier token
w = torch.tril(torch.ones(4, 4))
print(w / w.sum(dim=1, keepdim=True))
# row 0 = [1, 0, 0, 0]   row 3 = [.25, .25, .25, .25]

## 2. One attention head

Each token makes a query (what am I looking for), a key (what do I contain) and a value (what I hand over). `query . key` is the score, softmax turns scores into weights, and the output is a weighted sum of values.

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 6, 16)          # (B=2, T=6, C=16)
head = Head(n_embd=16, head_size=8, block_size=8)

out = head(x)
print(out.shape)                   # expect (2, 6, 8)

attn = head.last_attn[0]           # attention weights for the first sequence
print(attn.shape)                  # expect (6, 6)
print(attn.sum(dim=-1))            # expect all 1.0 (each row is a distribution)
print(torch.triu(attn, diagonal=1).abs().sum())   # expect 0: nothing attends to the future
print(attn.round(decimals=2))

## 3. Multi-head attention

Several small heads in parallel, each free to look for something different, then concatenated back to the full width.

In [ ]:
mha = MultiHeadAttention(n_embd=16, n_heads=4, block_size=8)
out = mha(x)

print(len(mha.heads), "heads, each", mha.heads[0].value.out_features, "wide")   # 4 heads, each 4 wide
print(out.shape)                   # expect (2, 6, 16): same shape in and out

## 4. The model

Token embedding + position embedding -> multi-head attention -> output layer. Untrained, it guesses at random, so the loss is near `ln(256) = 5.55`.

In [ ]:
data = load_data("02-bigram/data/input.txt")
train_data, val_data = split_data(data)

torch.manual_seed(0)
model = AttentionLanguageModel(VOCAB_SIZE, n_embd=64, n_heads=4, block_size=32).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")   # bigram had 65,536

from attention import get_batch
xb, yb = get_batch(train_data, block_size=32, batch_size=8, device=device)
logits, loss = model(xb, yb)
print(logits.shape)                # expect (8, 32, 256)
print(loss.item())                 # expect around 5.5 to 6

## 5. Train

Same training loop as step 02. The bigram stalled at ~2.47. Attention can use up to 32 tokens of context, so it should get clearly lower.

In [ ]:
torch.manual_seed(0)
model = AttentionLanguageModel(VOCAB_SIZE, n_embd=64, n_heads=4, block_size=32).to(device)
losses = train(model, train_data, steps=3000, batch_size=32, block_size=32,
               lr=3e-3, device=device)

print(f"first 100 steps avg: {sum(losses[:100]) / 100:.3f}")
print(f"last 100 steps avg:  {sum(losses[-100:]) / 100:.3f}")   # expect about 2.0 to 2.1

plt.plot(losses, alpha=0.3)
plt.axhline(2.47, color="red", linestyle="--", label="bigram (step 02)")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("attention model training loss")
plt.show()

## 6. Validation loss

Loss on text it never trained on. A small gap to the training loss is normal.

In [ ]:
from attention import get_batch

@torch.no_grad()
def eval_loss(model, data, n_batches=50):
    total = 0.0
    for _ in range(n_batches):
        xb, yb = get_batch(data, block_size=32, batch_size=32, device=device)
        _, loss = model(xb, yb)
        total += loss.item()
    return total / n_batches

print("train:", round(eval_loss(model, train_data), 3))
print("val:  ", round(eval_loss(model, val_data), 3))

## 7. Generate

Compare with the bigram: real words now appear (`will`, `for`, `love`), and speaker names like `DUKE`. Sentences still do not make sense, since this is one attention layer, no depth, and a short context.

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
out = model.generate(start, max_new_tokens=400)
print(decode(out[0].tolist()))

## 8. What did the heads learn to look at?

Row = the token doing the looking, column = the token it looks at. Brighter = more attention. The upper triangle is black because tokens cannot see the future. Each head learned its own pattern.

In [ ]:
text = "First Citizen:\nBefore we proceed"          # exactly 32 bytes
idx = torch.tensor([list(text.encode())], device=device)
print(idx.shape)                   # expect (1, 32)

model.eval()
with torch.no_grad():
    model(idx)

labels = ["\u23ce" if ch == "\n" else ch for ch in text]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for h, ax in enumerate(axes):
    ax.imshow(model.attn.heads[h].last_attn[0].cpu(), cmap="viridis")
    ax.set_xticks(range(len(labels)), labels, fontsize=7)
    ax.set_yticks(range(len(labels)), labels, fontsize=7)
    ax.set_title(f"head {h}")
plt.tight_layout()
plt.show()

## What is still missing

This is one attention layer, wired straight to the output. A real transformer block adds:
- **residual connections** so information and gradients flow easily when stacked
- **LayerNorm** to keep values stable
- an **MLP** after attention, so each token can think about what it gathered
- **many blocks** stacked, so later layers build on earlier ones

That is step 04.